In [1]:
# 01_detection.ipynb
# ------------------------------------------------------------------------------------
# Imports, configuration, grayscale plotting style, and output savers.
# Design choices: single-cell notebook; English comments only; figures in grayscale via
# seaborn; no captions; dpi=600; every figure saved as BOTH .png and .pdf; table outputs
# saved as .csv; file paths are never printed (anonymity).
# ------------------------------------------------------------------------------------
import os, copy, hashlib, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

# Reproducibility
GLOBAL_SEED = 2025
np.random.seed(GLOBAL_SEED)

# Paths (relative; intentionally never printed)
DATA_PATH   = os.path.join("..", "data", "criteo_clean.parquet")
FIG_DIR     = os.path.join("..", "results", "figures")
TBL_DIR     = os.path.join("..", "results", "tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TBL_DIR, exist_ok=True)

# Grayscale, publication-style seaborn theme
sns.set_theme(context="paper", style="whitegrid")
GRAYSCALE = ["#111111", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYSCALE))
plt.rcParams.update({
    "image.cmap": "gray",
    "axes.edgecolor": "#333333",
    "axes.labelcolor": "#000000",
    "text.color": "#000000",
    "xtick.color": "#000000",
    "ytick.color": "#000000",
    "font.size": 11,
    "savefig.bbox": "tight",
})

def save_fig(fig, name):
    """Save a figure as png and pdf at dpi=600. No captions, no path printing."""
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=600)
    plt.close(fig)

def save_table(df, name):
    """Save a results table as csv. No path printing."""
    df.to_csv(os.path.join(TBL_DIR, f"{name}.csv"), index=False)

# Experiment configuration
CFG = {
    "data": {
        "clean_path": DATA_PATH,
        # Only features observable at BIDDING TIME are used. The Criteo Attribution dataset's
        # click_pos/click_nb/cost/cpo/attribution are computed along the conversion PATH (e.g.
        # click_pos = "position of the click before a conversion", cpo = cost per ORDER), so they
        # are post-conversion and leak the label (each gives standalone AUROC ~1.0 vs conversion).
        # `click` is the stage-1 LABEL and must never be an input. We keep contextual categoricals,
        # the campaign id, and time_since_last_click (past-only recency).
        "cat_cols": ["campaign","cat1","cat2","cat3","cat4","cat5","cat6","cat7","cat8","cat9"],
        "dense_cols": ["time_since_last_click"],
        "stage_cols": ["click","conversion"],
        "max_vocab": 100000,
    },
    "split": {"use_existing_split": True, "train_max_day": 20, "valid_max_day": 24},
    "model": {"name":"leakfunnel","embed_dim":16,"hidden_dim":128,"n_heads":4,"n_layers":2,
              "dropout":0.1,"lambda_mono":0.1,"lambda_leak":0.1},
    # num_workers MUST be 0 on Windows + single-cell notebooks: spawned workers cannot
    # re-import classes defined in the notebook and will deadlock on the first batch.
    # We avoid DataLoader workers entirely by moving each split to the GPU up front
    # (see _tensors) and batching by index, so num_workers is not used for training.
    # amp (float16 mixed precision) is OFF by default: these models are small enough to be
    # fast in float32 on a consumer GPU, and AMP can produce NaN predictions on the
    # Transformer/ESMM paths. Turn it on only if you need the speed and have verified stability.
    "train": {"epochs":3,"batch_size":8192,"lr":1e-3,"weight_decay":1e-5,"num_workers":0,
              "device":"auto","seed":GLOBAL_SEED,"amp":False,"max_rows":None},
    "leakage": {"as_of_days":[15,20,25],"leak_strength":0.5,"leak_fraction":0.5,
                "noise_sd":0.3,"family":"both"},
    "eval": {"n_seeds":5,"bootstrap":1000},
}

# FAST switch: set FAST = True for a quick pass (subsampled rows, fewer epochs/seeds) to
# iterate or sanity-check; set FAST = False for the full, paper-grade run. One line to flip.
FAST = False
if FAST:
    CFG["train"]["max_rows"] = 300000
    CFG["train"]["epochs"] = 2
    CFG["eval"]["n_seeds"] = 2

def audit_input_leakage(cfg, label_col="conversion", thresh=0.99, n=300000):
    # Self-check: measure each MODEL INPUT feature's standalone AUROC against the label on a
    # subsample. A value near 1.0 means the feature essentially encodes the label (a
    # post-outcome/leaky column that must not be an input). Prints a warning per offender.
    # This is a guard against silently re-introducing label leakage into the baseline pipeline.
    from sklearn.metrics import roc_auc_score
    cols = cfg["data"]["cat_cols"] + cfg["data"]["dense_cols"] + [label_col]
    t = pq.read_table(cfg["data"]["clean_path"], columns=list(dict.fromkeys(cols)))
    m = t.num_rows
    idx = np.linspace(0, m-1, min(n, m)).astype(np.int64)
    df = t.take(idx).to_pandas()
    y = df[label_col].to_numpy()
    flagged = []
    for c in cfg["data"]["cat_cols"] + cfg["data"]["dense_cols"]:
        x = np.nan_to_num(df[c].to_numpy().astype(float))
        try:
            a = roc_auc_score(y, x); a = max(a, 1 - a)
        except Exception:
            a = float("nan")
        if np.isfinite(a) and a >= thresh:
            flagged.append((c, a)); print(f"LEAKAGE WARNING: input '{c}' has standalone AUROC {a:.4f} vs {label_col} -> remove it")
    if not flagged:
        print(f"input leakage audit: OK (no input feature exceeds AUROC {thresh} vs {label_col})")
    return flagged

def report_device():
    # Print the compute device once so it is obvious whether CUDA is actually used.
    # If this shows CPU on a GPU machine, the installed torch is a CPU build -> reinstall
    # torch with the CUDA index URL (see README). No file paths are printed.
    try:
        import torch
        if torch.cuda.is_available():
            print(f"device: cuda ({torch.cuda.get_device_name(0)}) | torch {torch.__version__}")
        else:
            print(f"device: cpu | torch {torch.__version__} | CUDA not available")
    except ImportError:
        print("torch not installed (only data/detection notebooks run without it)")
# ------------------------------------------------------------------------------------
# Controlled leakage-injection harness (exact ground truth).
# row family: feature = PIT_prior + s * future_label + noise           -> detectable
# campaign family: per-campaign rate including post-as-of conversions   -> diluted/hidden
# ------------------------------------------------------------------------------------
def inject_features(cfg, seed=7, force_leaky=False, strengths=None):
    lk = cfg["leakage"]
    t = pq.read_table(cfg["data"]["clean_path"],
        columns=["campaign","timestamp","conversion","conversion_timestamp","day","split"]).to_pandas()
    mr = cfg["train"].get("max_rows")
    if mr: t = t.iloc[:int(mr)].reset_index(drop=True)
    campaign=t["campaign"].to_numpy(); ts=t["timestamp"].to_numpy()
    conv=t["conversion"].to_numpy(); cts=t["conversion_timestamp"].to_numpy()
    rng=np.random.default_rng(seed)
    NC=int(campaign.max())+1 if len(campaign) else 2
    future_label=((conv==1)&(cts>=0)).astype(np.float64)
    families=["row","campaign"] if lk["family"]=="both" else [lk["family"]]
    out=pd.DataFrame(index=t.index); gt={}
    for tau in lk["as_of_days"]:
        cutoff=tau*86400; imp=ts<=cutoff
        num=np.zeros(NC); den=np.zeros(NC)
        np.add.at(den, campaign[imp], 1.0)
        np.add.at(num, campaign[(conv==1)&(cts>=0)&(cts<=cutoff)], 1.0)
        prior=np.divide(num,den,out=np.zeros(NC),where=den>0)[campaign]
        for fam in families:
            is_leaky=True if force_leaky else (rng.random()<lk["leak_fraction"])
            s=(strengths[tau] if strengths else lk["leak_strength"]) if is_leaky else 0.0
            noise=rng.normal(0, lk["noise_sd"], size=len(conv))
            if fam=="row":
                feat=prior + s*future_label + noise
            else:
                if is_leaky:
                    fn=np.zeros(NC); fd=np.zeros(NC)
                    np.add.at(fd, campaign[imp], 1.0)
                    np.add.at(fn, campaign[(conv==1)&(cts>=0)&imp], 1.0)
                    fut=np.divide(fn,fd,out=np.zeros(NC),where=fd>0)[campaign]
                    feat=(1-s)*prior + s*fut + noise
                else:
                    feat=prior+noise
            col=f"{fam}_feat_asof{tau}"
            out[col]=feat.astype(np.float32)
            gt[col]={"is_leaky":bool(is_leaky),"leak_strength":float(s),"family":fam,"as_of_day":int(tau)}
    out["conversion"]=conv; out["day"]=t["day"].to_numpy()
    return out, gt
# ------------------------------------------------------------------------------------
# Detection oracle (standalone AUROC vs as-of date) and baselines.
# ------------------------------------------------------------------------------------
def standalone_auroc(score, label):
    score=np.asarray(score,float); label=np.asarray(label)
    order=np.argsort(score,kind="mergesort"); r=np.empty(len(score)); r[order]=np.arange(1,len(score)+1)
    pos=label==1; npos=int(pos.sum()); nneg=int((~pos).sum())
    if npos==0 or nneg==0: return float("nan")
    return (r[pos].sum()-npos*(npos+1)/2)/(npos*nneg)

def auroc_vs_date(feature, label, day, as_of_days):
    out={}
    for tau in as_of_days:
        m=day<=tau
        if m.sum()<10: out[tau]=float("nan"); continue
        a=standalone_auroc(feature[m], label[m]); out[tau]=max(a,1-a)
    return out

def leak_score_from_curve(curve):
    vals=np.array([curve[t] for t in sorted(curve)],float); vals=vals[~np.isnan(vals)]
    if len(vals)==0: return 0.0
    peak=max(0.0, vals.max()-0.5); slope=max(0.0, vals[-1]-vals[0]) if len(vals)>=2 else 0.0
    return float(peak+slope)

def score_all_features(df, feature_cols, as_of_days, label_col="conversion", day_col="day"):
    label=df[label_col].to_numpy(); day=df[day_col].to_numpy(); res={}
    for c in feature_cols:
        curve=auroc_vs_date(df[c].to_numpy(), label, day, as_of_days)
        res[c]={"curve":curve,"z":leak_score_from_curve(curve)}
    return res

def correlation_shift_scores(df, feature_cols, as_of_days, label_col="conversion", day_col="day"):
    label=df[label_col].to_numpy().astype(float); day=df[day_col].to_numpy(); out={}
    for c in feature_cols:
        f=df[c].to_numpy().astype(float); corrs=[]
        for tau in as_of_days:
            m=day<=tau
            if m.sum()<10 or f[m].std()==0: continue
            corrs.append(abs(np.corrcoef(f[m],label[m])[0,1]))
        out[c]=float(max(corrs)-min(corrs)) if len(corrs)>=2 else 0.0
    return out

def evaluate_detection(scores, gt, score_key="z"):
    from sklearn.metrics import average_precision_score, roc_auc_score
    feats=[f for f in scores if f in gt]
    y=np.array([1 if gt[f]["is_leaky"] else 0 for f in feats])
    s=np.array([scores[f][score_key] if isinstance(scores[f],dict) else scores[f] for f in feats])
    m={"n_features":len(feats),"n_leaky":int(y.sum())}
    if y.sum() in (0,len(y)): m["auprc"]=float("nan"); m["auroc"]=float("nan"); m["precision"]=float("nan"); m["recall"]=float("nan"); return m
    m["auprc"]=float(average_precision_score(y,s)); m["auroc"]=float(roc_auc_score(y,s))
    thr=(s.max()+s.min())/2 if s.max()>s.min() else s.mean(); pred=(s>=thr).astype(int)
    tp=int(((pred==1)&(y==1)).sum()); fp=int(((pred==1)&(y==0)).sum()); fn=int(((pred==0)&(y==1)).sum())
    m["precision"]=tp/(tp+fp) if tp+fp else 0.0; m["recall"]=tp/(tp+fn) if tp+fn else 0.0
    return m

# ------------------------------------------------------------------------------------
# RQ1: leakage detection vs leak strength (oracle vs correlation-shift baseline).
# RQ2: oracle blind spot (row leak caught; campaign leak hidden).
# Tables saved to results/tables; grayscale figures to results/figures.
# ------------------------------------------------------------------------------------

# ---- RQ1 ----
rq1_rows = []
for strength in [0.0, 0.2, 0.4, 0.6, 0.8]:
    c = copy.deepcopy(CFG); c["leakage"]["leak_strength"] = strength
    inj, gt = inject_features(c, seed=7)
    fcols = list(gt.keys())
    sc = score_all_features(inj, fcols, CFG["leakage"]["as_of_days"])
    corr = correlation_shift_scores(inj, fcols, CFG["leakage"]["as_of_days"])
    mo = evaluate_detection(sc, gt, "z"); mc = evaluate_detection(corr, gt)
    rq1_rows.append({"leak_strength": strength,
                     "oracle_auprc": mo.get("auprc"), "oracle_recall": mo.get("recall"),
                     "corr_shift_auprc": mc.get("auprc")})
rq1 = pd.DataFrame(rq1_rows)
save_table(rq1, "rq1_detection")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(rq1["leak_strength"], rq1["oracle_auprc"], marker="o", color="#111111",
        linestyle="-", label="standalone-AUROC oracle")
ax.plot(rq1["leak_strength"], rq1["corr_shift_auprc"], marker="s", color="#999999",
        linestyle="--", label="correlation-shift baseline")
ax.set_xlabel("leak strength"); ax.set_ylabel("detection AUPRC")
ax.legend(frameon=False)
save_fig(fig, "rq1_detection_auprc")

# ---- RQ2 ----
c2 = copy.deepcopy(CFG); c2["leakage"]["family"] = "both"; c2["leakage"]["leak_strength"] = 0.6
inj2, gt2 = inject_features(c2, seed=7, force_leaky=True)
label = inj2["conversion"].to_numpy(); day = inj2["day"].to_numpy()
rq2_rows = []
for col, meta in gt2.items():
    m = day <= max(CFG["leakage"]["as_of_days"])
    a = standalone_auroc(inj2[col].to_numpy()[m], label[m]); a = max(a, 1 - a)
    rq2_rows.append({"feature": col, "family": meta["family"],
                     "leak_strength": meta["leak_strength"],
                     "standalone_auroc": round(a, 4), "oracle_flags_leak": bool(a > 0.6)})
rq2 = pd.DataFrame(rq2_rows)
save_table(rq2, "rq2_oracle_blindspot")

fig, ax = plt.subplots(figsize=(6, 4))
markers = {"row": "o", "campaign": "s"}; shades = {"row": "#111111", "campaign": "#999999"}
for fam in rq2["family"].unique():
    sub = rq2[rq2["family"] == fam].reset_index(drop=True)
    ax.scatter(range(len(sub)), sub["standalone_auroc"], marker=markers.get(fam, "o"),
               color=shades.get(fam, "#555555"), s=70, label=fam, edgecolor="#000000")
ax.axhline(0.6, linestyle="--", color="#000000", linewidth=1)
ax.set_xlabel("feature index"); ax.set_ylabel("standalone AUROC")
ax.legend(frameon=False)
save_fig(fig, "rq2_oracle_blindspot")

pd.concat([rq1.assign(table="RQ1"), rq2.assign(table="RQ2")], ignore_index=True)


,leak_strength,oracle_auprc,oracle_recall,corr_shift_auprc,table,feature,family,standalone_auroc,oracle_flags_leak
0,0.0,0.854167,0.75,0.566667,RQ1,NaN,NaN,NaN,NaN
1,0.2,0.950000,0.50,0.566667,RQ1,NaN,NaN,NaN,NaN
2,0.4,1.000000,0.50,0.608333,RQ1,NaN,NaN,NaN,NaN
3,0.6,1.000000,0.50,0.645833,RQ1,NaN,NaN,NaN,NaN
4,0.8,1.000000,0.50,0.770833,RQ1,NaN,NaN,NaN,NaN
5,0.6,NaN,NaN,NaN,RQ2,row_feat_asof15,row,0.9308,True
6,0.6,NaN,NaN,NaN,RQ2,campaign_feat_asof15,campaign,0.5410,False
7,0.6,NaN,NaN,NaN,RQ2,row_feat_asof20,row,0.9316,True
8,0.6,NaN,NaN,NaN,RQ2,campaign_feat_asof20,campaign,0.5425,False
9,0.6,NaN,NaN,NaN,RQ2,row_feat_asof25,row,0.9320,True
